In [1]:
import numpy as np
from time import sleep

from orchid.instrument import InstrumentAdapter
from orchid.controller import DataKind, Controller, Readout
from orchid.bench import Bench
from orchid.procedure import ErrorPolicy, MonitorProcedure, Procedure, MultiSweep, Sweep, WriteMode
from orchid.runner import ExperimentRunner
from orchid.plotting import LivePlotter, PlotSpec
from orchid.control_panel import ControlPanel

import plotly.io as pio
import plotly.graph_objects as go

In [2]:
class DummyVoltageSource:
    def __init__(self):
        self._voltage = 0.0

    @property
    def voltage(self):
        return self._voltage

    @voltage.setter
    def voltage(self, v):
        self._voltage = float(v)


class DummyLockin:
    """Lorentzian peak at V=0.5, with noise."""
    def __init__(self, source: DummyVoltageSource):
        self._src = source

    def read_x(self) -> float:
        v = self._src.voltage
        return 1.0 / (1.0 + ((v - 0.5) / 0.05) ** 2) + np.random.normal(0, 0.01)

    def read_y(self) -> float:
        v = self._src.voltage
        return -1.0 / (1.0 + ((v - 0.5) / 0.05) ** 2) - np.random.normal(0, 0.01)


class DummyGateSource:
    def __init__(self):
        self._vg1 = 0.0
        self._vg2 = 0.0

    @property
    def vg1(self): return self._vg1
    @vg1.setter
    def vg1(self, v): self._vg1 = v

    @property
    def vg2(self): return self._vg2
    @vg2.setter
    def vg2(self, v): self._vg2 = v


class DummyConductance:
    def __init__(self, gates: DummyGateSource):
        self._g = gates

    def read(self) -> float:
        # Gaussian peak in 2D gate space
        return np.exp(-((self._g.vg1 - 0.3)**2 + (self._g.vg2 - 0.7)**2) / 0.01) \
               + np.random.normal(0, 0.005)


class DummyVNA:
    def __init__(self):
        self._freq_center = 5e9  # Hz
        self.freqs = np.linspace(4e9, 6e9, 201)

    @property
    def freq_center(self): return self._freq_center
    @freq_center.setter
    def freq_center(self, f): self._freq_center = f

    def get_s21(self) -> np.ndarray:
        """Returns a resonance dip trace."""
        return -10 * np.exp(-((self.freqs - self._freq_center) / 50e6)**2) \
               + np.random.normal(0, 0.1, len(self.freqs))


In [3]:
# Instruments
vs = DummyVoltageSource()
li = DummyLockin(vs)
gs = DummyGateSource()
vna = DummyVNA()
cond = DummyConductance(gs)

# Context
bench = Bench(data_root="./data")
bench.add_instrument("vsrc", vs)
bench.add_instrument("gs", gs)
bench.add_controller("V0", instrument="vsrc", attr="voltage", unit="V", limits=[-1,1])
bench.add_controller("V1", instrument="gs", attr="vg1", unit="V", limits=[0,1])
bench.add_controller("V2", instrument="gs", attr="vg2", unit="V", limits=[0,1])


def Vr_setter(voltage: float):
    vs.voltage = voltage
    gs.vg1 = voltage
    

bench.add_controller_binding("Vr", ["V1", "V2"], limits=(0,1))


bench.add_readout("X", kind=DataKind.SCALAR, get_func=li.read_x, unit="V")
bench.add_readout("Y", kind=DataKind.SCALAR, get_func=li.read_y, unit="V")
bench.add_readout("S21", kind=DataKind.TRACE, get_func=vna.get_s21, shape=(201,), unit="dB")

def Vmag():
    return np.sqrt(li.read_x()**2 + li.read_y()**2)
bench.add_readout("Vmag", kind="scalar", get_func=Vmag, unit="V")
bench.snapshot(include_readouts=True)

Name    Type           Value                  Unit    Limits
------  -------------  ---------------------  ------  --------
V0      ctrl           0.0                    V       [-1, 1]
V1      ctrl           0.0                    V       [0, 1]
V2      ctrl           0.0                    V       [0, 1]
Vr      virtual        —                      V       (0, 1)
X       read / scalar  0.004888676893852425   V       N/A
Y       read / scalar  -0.016335783912592567  V       N/A
S21     read / trace   [...]                  dB      N/A
Vmag    read / scalar  0.0060075656395247685  V       N/A


In [4]:
# panel = ControlPanel(bench, port=8051, steps={"V0": 0.02, "V1": 0.01, "V2": 0.01, "Vr": 0.01})
# panel.start()

In [5]:
# panel.stop()

### **2D scan**

In [6]:
proc1 = Procedure(
    name="sweep",
    bench=bench,
    sweeps=[
        MultiSweep(["V1", "V2"], [np.linspace(0, 1, 3), np.linspace(0, 1, 3)]),
        Sweep("V0", np.linspace(0.1, 1, 101)),
    ],
    readouts=["X","Y","Vmag"],
    settle_time=0.1,          # 10ms settle after each set
    tags=["transport", "1d"],
    metadata={"field": "0T"},
    write_mode=WriteMode.SWEEPWISE
)

proc1.summary()

Experiment     sweep
Tags           transport, 1d
────────────────────────────────────────
Sweeps 2D      303 pts
[0] V1         0 → 1          3 pts    V
└── V2         0 → 1                   V
[1] V0         0.1 → 1        101 pts  V
────────────────────────────────────────
Readouts
X              scalar                  V
Y              scalar                  V
Vmag           scalar                  V
────────────────────────────────────────
Settings
write_mode     sweepwise
settle_time    100 ms
error_policy   stop_and_save
est. duration  ~30.3 s


In [7]:
plotter = LivePlotter([
    PlotSpec(x="V0", y=["X","Y"]),
    PlotSpec(x="V0", y="Vmag"),
    PlotSpec(x="V0", y="V1", z="Vmag", update_every="point")
    ]
)

plotter.prepare(proc1)

Live plot server started at http://localhost:8050


In [8]:
runner = ExperimentRunner()

In [9]:
runner.run(proc1, plotter=plotter)

sweep: 100%|██████████| 303/303 [00:31<00:00,  9.65pt/s]

Experiment 'sweep' completed. Data saved to: data/0004


In [10]:
plotter.stop()

Live plot server stopped.


In [11]:
LivePlotter.load("data/0003")

Live plot server started at http://localhost:8052


### **Time Series**

In [14]:
monitor = MonitorProcedure(
    name="stability_check",
    bench=bench,
    readouts=["X","Y","Vmag"],
    interval=0.2,       # 1 second between reads
    duration=None,     # 1 hour
    tags=["stability"],
)
runner = ExperimentRunner()
plotter = LivePlotter([PlotSpec(x="_time", y=["X","Y","Vmag"], plot_type='line')], height=600, width=800)

monitor.summary()

Monitor     stability_check
Tags        stability
────────────────────────────────
Readouts
X           scalar             V
Y           scalar             V
Vmag        scalar             V
────────────────────────────────
Settings
interval    0.2 s
duration    unlimited
chunk_size  256


In [15]:
plotter.prepare(monitor)

Live plot server started at http://localhost:8050


In [16]:
runner.run_monitor(monitor, plotter=plotter, background=True)

Monitor 'stability_check' running in background. Use runner.stop_monitor() to stop.


In [8]:
bench["V0"] = 0.4

In [17]:
plotter.stop()

Live plot server stopped.


In [18]:
bench.snapshot()

Name    Type     Value    Unit    Limits
------  -------  -------  ------  --------
V0      ctrl     0.46     V       [-1, 1]
V1      ctrl     0.63     V       [0, 1]
V2      ctrl     0.63     V       [0, 1]
Vr      virtual  —        V       (0, 1)
